In [1]:
!pip install torch


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install numpy
!pip install sentence_transformers


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:

import csv
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
import sys
import os

sys.path.append("/workspaces/hallucilation_in_llm")



from model.hf_model import HFModel
from pipeline.runner import PipelineRunner
from evaluation.evaluator import Evaluator
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider
from uncertainty.black_uncertainty import BlackBoxUncertainty
from uncertainty.graybox_uncertainty import GrayBoxUncertainty
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.semantic_uncertainty import EnsembleSemanticUncertainty
from stat_metrics import StatisticalAnalyzer
from calibration import CalibrationMetrics



prompts_en = [
    "What is the capital of France?",
    "Explain photosynthesis.",
    "Who painted the Mona Lisa?",
    "Define black holes.",
    "What is quantum entanglement?"
]

ground_truth_en = [
    "Paris",
    "Photosynthesis is the process by which plants convert light into chemical energy.",
    "Leonardo da Vinci",
    "A black hole is a region in space with gravity so strong that nothing can escape.",
    "Quantum entanglement is a phenomenon where particles remain connected regardless of distance."
]



model = HFModel(
    model_name="gpt2",  
    hf_token="--"  


uncertainty_modules = {
    "blackbox": lambda output: BlackBoxUncertainty(output.responses),
    "graybox": lambda output: GrayBoxUncertainty(output.responses, output.log_probs),
    "whitebox": lambda output: WhiteBoxUncertainty(output.logits, output.token_ids, output.responses),
    "semantic": lambda output: EnsembleSemanticUncertainty(output.responses, language="en")
}

final_score_calc = FinalScore()
evaluator = Evaluator()

decider = HallucinationDecider(
    thresholds={
        "entropy": 5.0,
        "confidence": 0.35,
        "consistency": 0.5,
        "risk": 2.5
    }
)

pipeline = PipelineRunner(model, uncertainty_modules, evaluator, decider)



results = []

for prompt in prompts_en:
    res = pipeline.run(prompt)
    results.append(res)

    print("="*60)
    print("PROMPT:", prompt)
    print("RESPONSES:", res["responses"])
    print("UNCERTAINTY:", res["uncertainty"])
    print("FINAL SCORE:", res["evaluation"]["final_score"])
    print("DECISION:", res["decision"])



class GroundTruthSemanticEvaluator:
    def __init__(self, ground_truth, model_path="./models/paraphrase-multilingual-MiniLM-L12-v2", device=None):
        self.ground_truth = ground_truth
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(model_path, device=self.device)

        # GT embedding tek sefer hesaplanır
        self.gt_embeddings = self.model.encode(
            ground_truth,
            convert_to_tensor=True,
            normalize_embeddings=True
        )

    def compute_similarity(self, responses):
        sim_scores = []
        for idx, resp_list in enumerate(responses):

            if isinstance(resp_list, str):
                resp_list = [resp_list]
            resp_embs = self.model.encode(
                resp_list,
                convert_to_tensor=True,
                normalize_embeddings=True
            )
            sims = torch.nn.functional.cosine_similarity(
                resp_embs,
                self.gt_embeddings[idx].unsqueeze(0)
            )
            sim_scores.append(float(sims.mean()))
        return sim_scores

responses_list = [r["responses"] for r in results]

gt_eval = GroundTruthSemanticEvaluator(ground_truth_en)
similarity_scores = gt_eval.compute_similarity(responses_list)



threshold = 0.7
labels = np.array([1 if sim < threshold else 0 for sim in similarity_scores])
final_scores = np.array([r["evaluation"]["final_score"] for r in results])


pearson, spearman = StatisticalAnalyzer.correlation(final_scores, similarity_scores)
auroc = StatisticalAnalyzer.auroc(final_scores, labels)
pr_auc = StatisticalAnalyzer.pr_auc(final_scores, labels)
brier = CalibrationMetrics.brier_score(final_scores, labels)
ece = CalibrationMetrics.expected_calibration_error(final_scores, labels)

print("\n===== GLOBAL METRICS =====")
print("Pearson:", pearson)
print("Spearman:", spearman)
print("AUROC:", auroc)
print("PR-AUC:", pr_auc)
print("Brier Score:", brier)
print("ECE:", ece)


csv_columns = [
    "Prompt",
    "Responses",
    "Blackbox",
    "Graybox",
    "Whitebox",
    "Semantic",
    "Final_Score",
    "Decision",
    "Semantic_Similarity",
    "Label",
    "Pearson",
    "Spearman",
    "AUROC",
    "PR_AUC",
    "Brier",
    "ECE"
]

csv_file = "pipeline_results_en.csv"

with open(csv_file, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
    writer.writeheader()
    for i, r in enumerate(results):
        writer.writerow({
            "Prompt": prompts_en[i],
            "Responses": " || ".join(r["responses"]),
            "Blackbox": r["uncertainty"].get("blackbox_black_entropy"),
            "Graybox": r["uncertainty"].get("graybox_gray_entropy"),
            "Whitebox": r["uncertainty"].get("whitebox_white_entropy"),
            "Semantic": r["uncertainty"].get("semantic_semantic_uncertainty"),
            "Final_Score": float(final_scores[i]),
            "Decision": r["decision"],
            "Semantic_Similarity": float(similarity_scores[i]),
            "Label": int(labels[i]),
            "Pearson": pearson,
            "Spearman": spearman,
            "AUROC": auroc,
            "PR_AUC": pr_auc,
            "Brier": brier,
            "ECE": ece
        })

print(f"\nAll results saved to '{csv_file}'")

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HfHubHTTPError: (Amz CF ID: tlm6SVlme9tTz6xf2-3kj2_kLQ3IVC279txcdmkEnSLm7hkkdPVnHg==)

403 Forbidden: None.
Cannot access content at: https://huggingface.co/api/whoami-v2.
Make sure your token has the correct permissions.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from model.hf_model import HFModel


hf_token = "--"  

model = HFModel(model_name="gpt2", hf_token=hf_token)   


output = model.generate("What is the capital of France?", num_samples=2)
print(output.responses)

HfHubHTTPError: (Amz CF ID: xmYo6pvNOBZODosDT4BFeKv6kD-5z3CQjdhEHSptSfUHl7MYk5VC8g==)

403 Forbidden: None.
Cannot access content at: https://huggingface.co/api/whoami-v2.
Make sure your token has the correct permissions.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/home/burak/.cache/huggingface/transformers/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)

inputs = tokenizer("Hello world!", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=10)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/home/burak/.cache/huggingface/transformers/gpt2'. Use `repo_type` argument if needed.